In [1]:
# %%
#
# Cell 1: Initial Setup
#
import pandas as pd
import numpy as np
import torch
import random
import os

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding
)

from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    matthews_corrcoef, balanced_accuracy_score,
    cohen_kappa_score, jaccard_score, hamming_loss,
    confusion_matrix
)



/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-06-28 18:57:11.920583: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-28 18:57:12.864400: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
202

In [2]:
# %%
#
# Cell 2: W&B and Hugging Face Login
#
import wandb
import optuna
import huggingface_hub

import os
os.environ["WANDB_PROJECT"] = "distilbert_gendered"

# NOTE: Replace with your actual keys or use environment variables
# wandb.login(key="YOUR_WANDB_KEY")
# huggingface_hub.login(token="YOUR_HF_TOKEN")


In [3]:
# %%
#
# Cell 3: Model Configuration
#
model_name = "distilbert-base-uncased"
model_cache_path = "../scratch/cache/distilbert_gendered"



In [4]:
# %%
#
# Cell 4: Data Preparation
#
# Ensure 'data/combined_letters_gendered.csv' exists at the specified path
df = pd.read_csv("data/combined_letters_gendered.csv")[["full_text", "label"]].dropna()
df["label"] = df["label"].astype(int)

# Perform train-test split, stratifying by label to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    df["full_text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
)

# The tokenizer is always loaded from the base model
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_cache_path)



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
# %%
#
# Cell 5: Tokenization
#
# This function prepares your text data for the model by converting it into token IDs
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding=False,
        max_length=512
    )
    tokens["labels"] = example["label"]
    return tokens

# Convert pandas Series to Hugging Face Dataset objects, then map the tokenization function
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
test_dataset = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

tokenized_train = train_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_test = test_dataset.map(tokenize, batched=True).remove_columns(["text"])

# Data Collator: Dynamically pads input sequences to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)



Map:   0%|          | 0/7189 [00:00<?, ? examples/s]

Map:   0%|          | 0/1798 [00:00<?, ? examples/s]

In [6]:
# %%
#
# Cell 6: Metrics Function
#
# Compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = logits.argmax(-1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)

    print("Confusion Matrix:", confusion_matrix(labels, preds))

    return {
        "f1_score": f1,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "mcc": matthews_corrcoef(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "cohen_kappa": cohen_kappa_score(labels, preds),
        "jaccard": jaccard_score(labels, preds, average="macro"),
        "hamming_loss": hamming_loss(labels, preds)
    }



In [7]:
# %%
#
# Cell 7: Model Initialization for Hyperparameter Optimization
#
def model_init(trial=None):
    # Load the base pre-trained model (DistilBERT)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label={0: "female", 1: "male"},
        label2id={"female": 0, "male": 1},
        cache_dir=model_cache_path,
        device_map="auto"
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    return model



In [8]:
# %%
#
# Cell 8: Optuna Hyperparameter Space
#
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 10),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.03, step=0.01),
    }



In [9]:
# %%
#
# Cell 9: Trainer for Hyperparameter Optimization
#
training_args_for_hpo = TrainingArguments(
    output_dir="../scratch/hpo_results_distilbert_gendered",
    per_device_eval_batch_size=32,
    fp16=True,
    save_strategy="no",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=False,
    eval_strategy="epoch",
)

trainer = Trainer(
    model_init=model_init,
    args=training_args_for_hpo,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)



/tmp/ipykernel_2358966/2513378983.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# %%
#
# Cell 10: Run Hyperparameter Search
#
best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=20,
    compute_objective=lambda metrics: metrics["eval_f1_score"]
)

print("Best run details:")
print(best_run)

# Access the best hyperparameters found by Optuna
best_hps = best_run.hyperparameters
print("Best Hyperparameters Found:")
for hp, value in best_hps.items():
    print(f"  {hp}: {value}")

# W&B will provide a URL to the best run in its logs.
if hasattr(best_run, 'url'):
    print(f"Find the best run and explore all trials in W&B at: {best_run.url}")



[I 2025-06-28 18:58:51,771] A new study created in memory with name: no-name-3a3f1a3c-29e1-47aa-8521-609042e3932d
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: mtwesley to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,F1 Score,Accuracy,Precision,Recall,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.000200,0.000474,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.059200,1697.565000,53.816000
2,0.022900,0.033350,0.995005,0.994994,0.995074,0.994994,0.988414,0.996374,0.988347,0.988423,0.005006,1.049800,1712.649000,54.294000
3,0.000000,0.009654,0.998888,0.998888,0.998892,0.998888,0.997405,0.999194,0.997401,0.997405,0.001112,1.052500,1708.335000,54.157000
4,0.000000,0.009866,0.998333,0.998331,0.998340,0.998331,0.996112,0.998791,0.996104,0.996113,0.001669,1.053600,1706.510000,54.100000
5,0.000100,0.011351,0.998888,0.998888,0.998892,0.998888,0.997405,0.999194,0.997401,0.997405,0.001112,1.056100,1702.447000,53.971000
6,0.000000,0.011596,0.998888,0.998888,0.998892,0.998888,0.997405,0.999194,0.997401,0.997405,0.001112,1.056400,1702.002000,53.957000
7,0.000000,0.011795,0.998888,0.998888,0.998892,0.998888,0.997405,0.999194,0.997401,0.997405,0.001112,1.056900,1701.195000,53.931000
8,0.000000,0.011886,0.998888,0.998888,0.998892,0.998888,0.997405,0.999194,0.997401,0.997405,0.001112,1.055200,1703.902000,54.017000


Confusion Matrix: [[ 557    0]
 [   0 1241]]
Confusion Matrix: [[ 557    0]
 [   9 1232]]
Confusion Matrix: [[ 557    0]
 [   2 1239]]
Confusion Matrix: [[ 557    0]
 [   3 1238]]
Confusion Matrix: [[ 557    0]
 [   2 1239]]
Confusion Matrix: [[ 557    0]
 [   2 1239]]
Confusion Matrix: [[ 557    0]
 [   2 1239]]
Confusion Matrix: [[ 557    0]
 [   2 1239]]


[I 2025-06-28 19:01:07,184] Trial 0 finished with value: 0.9988882011474317 and parameters: {'learning_rate': 2.3484517021126504e-05, 'num_train_epochs': 8, 'per_device_train_batch_size': 8, 'weight_decay': 0.03}. Best is trial 0 with value: 0.9988882011474317.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


eval/accuracy,█▁▆▆▆▆▆▆
eval/balanced_accuracy,█▁▆▆▆▆▆▆
eval/cohen_kappa,█▁▆▆▆▆▆▆
eval/f1_score,█▁▆▆▆▆▆▆
eval/hamming_loss,▁█▃▃▃▃▃▃
eval/jaccard,█▁▆▆▆▆▆▆
eval/loss,▁█▃▃▃▃▃▃
eval/mcc,█▁▆▆▆▆▆▆
eval/precision,█▁▆▆▆▆▆▆
eval/recall,█▁▆▆▆▆▆▆
eval/runtime,█▁▃▄▆▆▆▅


...

Epoch,Training Loss,Validation Loss,F1 Score,Accuracy,Precision,Recall,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.018500,0.169421,0.979242,0.979422,0.979856,0.979422,0.951987,0.967776,0.951080,0.952407,0.020578,1.052100,1708.916000,54.176000


Confusion Matrix: [[ 522   35]
 [   2 1239]]


[I 2025-06-28 19:12:40,393] Trial 18 pruned. 
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


eval/accuracy,▁
eval/balanced_accuracy,▁
eval/cohen_kappa,▁
eval/f1_score,▁
eval/hamming_loss,▁
eval/jaccard,▁
eval/loss,▁
eval/mcc,▁
eval/precision,▁
eval/recall,▁
eval/runtime,▁


Epoch,Training Loss,Validation Loss


In [ ]:
# %%
#
# Cell 11: Final Model Retraining
#
# Re-training the model with the best hyperparameters
best_hps = best_run.hyperparameters

# Re-initialize the base model
final_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "female", 1: "male"},
    label2id={"female": 0, "male": 1},
    cache_dir=model_cache_path,
    device_map="auto"
)

final_model.config.pad_token_id = tokenizer.pad_token_id

# Define TrainingArguments for the final training run
final_model_output_dir = "../scratch/final_distilbert_gendered_model"
final_training_args = TrainingArguments(
    output_dir=final_model_output_dir,
    per_device_train_batch_size=best_hps["per_device_train_batch_size"],
    per_device_eval_batch_size=best_hps["per_device_train_batch_size"],
    num_train_epochs=best_hps["num_train_epochs"],
    learning_rate=best_hps["learning_rate"],
    weight_decay=best_hps["weight_decay"],
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    eval_strategy="epoch",
    save_total_limit=1,
    run_name="final_distilbert_model_training"
)

# Initialize the Trainer for final training
final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train the final model
final_trainer.train()

# Evaluate the final (best) model
eval_results = final_trainer.evaluate()
print("Final Model Evaluation Results:")
print(eval_results)

# Save the final trained model and tokenizer
final_trainer.save_model(final_model_output_dir)
tokenizer.save_pretrained(final_model_output_dir)

print(f"Final model and tokenizer saved to: {final_model_output_dir}")
